# KWISMO — Notebook 04 : Collecte de Données (Scraping + OCR) & Graphiques Suivis

Ce notebook permet d'exécuter et de visualiser les métriques de la collecte de données :
1. **Scraping Web Dynamique** (`src.data.scrape`).
2. **Extraction OCR** (`src.data.ocr`) avec EasyOCR.
3. **Visualisations Graphiques de Collecte** (Volume par domaine, Taux de succès OCR, Évolution de la collecte).
4. **Sauvegarde & Synchronisation Google Drive**.

Le projet exige **Python 3.13** (voir `src/__init__.py`).

In [ ]:
# Détection automatique de l'environnement (Local, Google Colab, Kaggle Notebooks)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            print("Connexion automatique à Google Drive...")
            drive.mount('/content/drive')
        except Exception as err:
            print(f"Montage Google Drive recommandé : {err}")
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

In [ ]:
# Configuration du dossier de travail sur Cloud (Colab / Kaggle) et Local
if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
elif ON_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /kaggle/working/kwismo
    else:
        !git -C /kaggle/working/kwismo fetch && git -C /kaggle/working/kwismo reset --hard origin/main
else:
    current = Path.cwd()
    PROJECT_DIR = current
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            PROJECT_DIR = candidate
            break
        elif (candidate / "kwismo-ai" / "src").exists():
            PROJECT_DIR = candidate / "kwismo-ai"
            break

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
print("Dossier racine du projet kwismo-ai :", PROJECT_DIR)

In [ ]:
# Sélection stricte de l'environnement virtuel local (.venv) ou cloud (.venv313)
VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB or ON_KAGGLE:
    python313_bin = VENV_DIR / "bin" / "python"
    if not python313_bin.exists():
        print("Installation de Python 3.13 et création de l'environnement .venv313...")
        !apt-get update -y
        !apt-get install -y software-properties-common
        !add-apt-repository -y ppa:deadsnakes/ppa
        !apt-get update -y
        !apt-get install -y python3.13 python3.13-venv python3.13-dev
        !{VENV_DIR}/bin/pip install --upgrade pip
        !{VENV_DIR}/bin/pip install -r requirements.txt
        !{VENV_DIR}/bin/python -m playwright install --with-deps chromium
    PYTHON_BIN = str(python313_bin)
    site_packages = VENV_DIR / "lib" / f"python3.13" / "site-packages"
    if site_packages.exists() and str(site_packages) not in sys.path:
        sys.path.insert(0, str(site_packages))
else:
    local_venv_win = PROJECT_DIR / ".venv" / "Scripts" / "python.exe"
    local_venv_nix = PROJECT_DIR / ".venv" / "bin" / "python"
    
    if local_venv_win.exists():
        PYTHON_BIN = str(local_venv_win)
        site_pkgs = PROJECT_DIR / ".venv" / "Lib" / "site-packages"
        if site_pkgs.exists() and str(site_pkgs) not in sys.path:
            sys.path.insert(0, str(site_pkgs))
    elif local_venv_nix.exists():
        PYTHON_BIN = str(local_venv_nix)
        site_pkgs = PROJECT_DIR / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
        if site_pkgs.exists() and str(site_pkgs) not in sys.path:
            sys.path.insert(0, str(site_pkgs))
    else:
        PYTHON_BIN = sys.executable

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run_module(module: str) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

ver_proc = subprocess.run([PYTHON_BIN, "--version"], capture_output=True, text=True)
print("Interprète Python configuré :", PYTHON_BIN)
print("Version vérifiée :", ver_proc.stdout.strip() or ver_proc.stderr.strip())
print("Accès sys.path configuré pour le projet et .venv : OK")

## 1. Scraping Web Dynamique

Découverte de pages via mots-clés, extraction du texte principal, téléchargement des images des articles pour OCR, et dédoublonnage atomique (SQLite).

In [ ]:
run_module("src.data.scrape")

## 2. Extraction OCR sur les Images Scrapées

Extraction du texte des captures et images d'articles via EasyOCR, enregistrement dans `messages.jsonl`.

In [ ]:
run_module("src.data.ocr")

## 3. Visualisations Graphiques de la Collecte & OCR

Génération de **3 graphiques visuels** retraçant la répartition par domaine web, le taux d'OCR et l'évolution du volume collecté.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="pastel")

scraped_path = PROJECT_DIR / "data" / "raw" / "scraped" / "messages.jsonl"
kwismo_path = PROJECT_DIR / "data" / "raw" / "kwismo_data" / "messages.jsonl"
target_file = scraped_path if scraped_path.exists() else kwismo_path

records = []
if target_file.exists():
    with open(target_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

df_scraped = pd.DataFrame(records)

if not df_scraped.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graphique 1 : Répartition des articles collectés par domaine web
    if "source" in df_scraped.columns:
        top_sources = df_scraped["source"].value_counts().head(8)
        sns.barplot(x=top_sources.values, y=top_sources.index, ax=ax1, palette="crest")
        ax1.set_title("1. Top des Domaines Web Scrapés", fontsize=12, fontweight="bold")
        ax1.set_xlabel("Nombre de pages / articles")
        
    # Graphique 2 : Proportion des types d'entrées (Texte vs Image OCR)
    if "type" in df_scraped.columns:
        types_counts = df_scraped["type"].value_counts()
        ax2.pie(types_counts, labels=types_counts.index, autopct="%1.1f%%", colors=["#3A86FF", "#FF006E"], startangle=120, wedgeprops=dict(width=0.4, edgecolor='w'))
        ax2.set_title("2. Taux d'Extraction OCR vs Texte Brut", fontsize=12, fontweight="bold")
        
    plt.tight_layout()
    plt.show()
else:
    print("ℹAucune donnée de scraping disponible pour générer les graphiques.")

## 4. Sauvegarde & Synchronisation des Données Scrapées sur Google Drive

Copie des fichiers scrapés (`messages.jsonl`, `images/`, `metrics/`, `known_domains.json`) directement dans votre dossier Google Drive (`/content/drive/MyDrive/kwismo_data`).

In [ ]:
import shutil

SAVE_METHOD = "drive"  # Options : "drive" ou "git" / "local"

if ON_COLAB and SAVE_METHOD == "drive":
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
        
    dest = Path("/content/drive/MyDrive/kwismo_data")
    dest.mkdir(parents=True, exist_ok=True)
    
    if Path("data/raw/scraped/messages.jsonl").exists():
        shutil.copy("data/raw/scraped/messages.jsonl", dest / "messages.jsonl")
    if Path("data/raw/kwismo_data/messages.jsonl").exists():
        shutil.copy("data/raw/kwismo_data/messages.jsonl", dest / "messages.jsonl")
    if Path("data/interim/known_domains.json").exists():
        shutil.copy("data/interim/known_domains.json", dest / "known_domains.json")
    if Path("data/raw/scraped/images").exists():
        shutil.copytree("data/raw/scraped/images", dest / "images", dirs_exist_ok=True)
    if Path("data/interim/metrics").exists():
        shutil.copytree("data/interim/metrics", dest / "metrics", dirs_exist_ok=True)
        
    print(f"Sauvegarde sur Google Drive terminée avec succès : {dest}")
elif SAVE_METHOD == "git":
    !git add data/raw/scraped/*.jsonl data/interim/metrics/ data/interim/known_domains.json
    !git commit -m "Collecte de données (scraping session)"
    print("Données committées sur Git.")
else:
    print(f"Collecte achevée. Données conservées localement dans {PROJECT_DIR}/data/raw/scraped/")